In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
!curl -LO https://raw.githubusercontent.com/speakleash/bielik-tools/refs/heads/main/tools/bielik_advanced_chat_template.jinja

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3977  100  3977    0     0   8171      0 --:--:-- --:--:-- --:--:--  8166


In [ ]:
import getpass
import os

# ACCEPT AGREEMENT: https://huggingface.co/speakleash/Bielik-4.5B-v3.0-Instruct
# AND PUT YOUR HF_TOKEN TO DOWNLOAD MODEL
print('Enter HF_TOKEN:')
os.environ["HF_TOKEN"] = getpass.getpass()

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import os


max_seq_length = 2048
dtype = None
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "speakleash/Bielik-4.5B-v3.0-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

del os.environ["HF_TOKEN"]


unsloth_template = open("bielik_advanced_chat_template.jinja","r").read()
unsloth_eos_token = "<|im_end|>"

tokenizer = get_chat_template(
    tokenizer,
    chat_template = (unsloth_template, unsloth_eos_token,),
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
    map_eos_token = True,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.7.0+cu126 with CUDA 1206 (you have 2.6.0+cu124)
    Python  3.11.12 (you have 3.11.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.10: Fast Llama patching. Transformers: 4.52.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-59): 60 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=True)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=True)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=True)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
      )
    )
    (

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                },
                "required": ["location"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_n_day_weather_forecast",
            "description": "Get an N-day weather forecast",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "num_days": {
                        "type": "integer",
                        "description": "The number of days to forecast",
                    }
                },
                "required": ["location", "num_days"]
            },
        }
    },
]

messages = [
    {"role": "user", "content": "A jaka będzie pogoda przez najbliższe 3 dni w Kielcach ?"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt", enable_thinking=True, tools=tools).to("cuda")
output_ids = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True)
input_size = inputs[0].shape[0]
output_text = tokenizer.decode(output_ids[0][input_size:], skip_special_tokens=False)
output_text

'<tool_call> {"name": "get_n_day_weather_cast", "arguments": {"location": "Kielce", "num_days": 3}}</tool_call><|im_end|>'